# unbroadcast-pattern — ex1: unbroadcast: sum out leading and size-1 broadcast axes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbroadcast-pattern`. Running the final beacon cell reports progress against the `Backprop: Unbroadcast pattern` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Unbroadcast pattern` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbroadcast-pattern`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbroadcast-pattern"
DD_SUBTOPIC = "Backprop: Unbroadcast pattern"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Unbroadcast pattern — quick refresher

PyTorch's elementwise ops broadcast — `x + y` may yield a result with more dims than either input. The reverse pass has the opposite problem: `grad_out` has the broadcast (output) shape, but `dL/dx` must have `x`'s *original* shape. Solution: **sum out the broadcast axes**.

Two cases to handle:
- **Leading new axes.** If `x.shape == (3, 4)` and `out.shape ==   (2, 3, 4)`, broadcasting added a leading dim of size 2. Sum it out:
  `grad_x = grad_out.sum(dim=0)`.
- **Size-1 dims that got expanded.** If `x.shape == (1, 4)` and   `out.shape == (3, 4)`, broadcasting expanded `x`'s leading dim. Sum it   out with `keepdim=True` to preserve the size-1 axis:   `grad_x = grad_out.sum(dim=0, keepdim=True)`.

Canonical recipe:
```python
def unbroadcast(grad, x):
    # 1. peel leading axes
    while grad.ndim > x.ndim:
        grad = grad.sum(dim=0)
    # 2. peel expanded size-1 axes
    for i, size in enumerate(x.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad
```

Every binary back fn wraps its result in `unbroadcast(..., x)` so the returned grad always matches the parent's stored shape.

### Exercise 1 — unbroadcast: sum out leading and size-1 broadcast axes

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the unbroadcast pattern (peel leading axes, then sum-with-keepdim across size-1 expanded axes) to restore grad to the pre-broadcast input shape.
> Keywords: unbroadcast, broadcasting, sum-axes, keepdim, binary-back
> ```

**KCs targeted:** `unbroadcast-pattern`, `chain-rule-elementwise`

Implement `unbroadcast(grad, original)`. Forward broadcasting can expand a tensor `original.shape` to a bigger `grad.shape` in two ways:

- **(A) leading new axes** — broadcasting added dims to the LEFT. Example: `original.shape=(3,4)`, `grad.shape=(2,3,4)` → leading dim of size 2 was added. Sum it out: `grad.sum(dim=0)`.
- **(B) size-1 axes that got expanded** — `original` had a size-1 axis that broadcasting expanded. Example: `original.shape=(1,4)`, `grad.shape=(3,4)` (after step A leaves grad shape `(3,4)`)... wait that's leading. Try: `original.shape=(3,1,4)`, `grad.shape=(3,5,4)` → axis 1 was size 1, got expanded to 5. Sum it out **with keepdim=True**: `grad.sum(dim=1, keepdim=True)`.

Recipe (do in this order):

1. While `grad.ndim > original.ndim`: `grad = grad.sum(dim=0)`. (Peels the leading axes.)
2. For each axis `i` in `original.shape`: if `original.shape[i] == 1` AND `grad.shape[i] != 1`, do `grad = grad.sum(dim=i, keepdim=True)`. (Collapses each expanded size-1 axis back to 1.)

Final result: `grad.shape == original.shape`. The function is the RIGHT-INVERSE of broadcasting in the sense that summing out the broadcasted axes recovers shape compatibility.

Inputs are plain `torch.Tensor`. No autograd. Return a float tensor with the same shape as `original`.

In [ ]:
def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    """Sum out axes broadcasting added/expanded so grad.shape matches original.shape."""
    raise NotImplementedError()


def _test_ex1():
    # --- no broadcasting: shapes already match → grad unchanged (value+shape) ---
    g = t.ones(3, 4)
    x = t.zeros(3, 4)
    out = unbroadcast(g, x)
    assert out.shape == (3, 4), f'identity shape: {out.shape}'
    assert t.allclose(out, t.ones(3, 4)), 'identity values'

    # --- case A: leading axes added ---
    # original.shape=(3,4), grad.shape=(2,3,4) → sum dim=0 → (3,4)
    g = t.ones(2, 3, 4)
    x = t.zeros(3, 4)
    out = unbroadcast(g, x)
    assert out.shape == (3, 4), f'leading axes shape: {out.shape}'
    assert t.allclose(out, t.full((3, 4), 2.0)), f'sum value wrong: {out}'

    # --- case A x2: TWO leading axes added ---
    g = t.ones(5, 2, 3, 4)
    x = t.zeros(3, 4)
    out = unbroadcast(g, x)
    assert out.shape == (3, 4)
    assert t.allclose(out, t.full((3, 4), 10.0)), 'sum across 5*2=10 wrong'

    # --- case B: size-1 axis got expanded ---
    # original.shape=(1,4), grad.shape=(3,4) — but ndim already matches
    # (this is pure case-B: leading-axes peel does nothing, size-1 collapse fires)
    g = t.ones(3, 4)
    x = t.zeros(1, 4)
    out = unbroadcast(g, x)
    assert out.shape == (1, 4), f'size-1 axis shape: {out.shape}'
    assert t.allclose(out, t.full((1, 4), 3.0)), f'size-1 value wrong: {out}'

    # --- case B: middle size-1 axis ---
    # original.shape=(3,1,4), grad.shape=(3,5,4) — middle axis expanded 1→5
    g = t.ones(3, 5, 4)
    x = t.zeros(3, 1, 4)
    out = unbroadcast(g, x)
    assert out.shape == (3, 1, 4), f'middle size-1 shape: {out.shape}'
    assert t.allclose(out, t.full((3, 1, 4), 5.0))

    # --- combined case A + B ---
    # original.shape=(1,4), grad.shape=(2,3,4)
    # step 1 peels leading axis → (3,4); step 2 collapses size-1 axis-0 → (1,4)
    g = t.ones(2, 3, 4)
    x = t.zeros(1, 4)
    out = unbroadcast(g, x)
    assert out.shape == (1, 4), f'A+B shape: {out.shape}'
    assert t.allclose(out, t.full((1, 4), 6.0)), f'A+B value (2*3=6): {out}'

    # --- scalar case: original is 0-D, grad is anything ---
    g = t.ones(2, 3)
    x = t.tensor(0.0)
    out = unbroadcast(g, x)
    assert out.shape == (), f'scalar shape: {out.shape}'
    assert t.allclose(out, t.tensor(6.0)), f'scalar value: {out}'

    # --- value correctness on non-uniform grad ---
    g = t.tensor([[1.0, 2.0, 3.0, 4.0],
                  [5.0, 6.0, 7.0, 8.0],
                  [9.0, 10.0, 11.0, 12.0]])
    x = t.zeros(1, 4)
    out = unbroadcast(g, x)
    assert out.shape == (1, 4)
    assert t.allclose(out, t.tensor([[15.0, 18.0, 21.0, 24.0]])), (
        f'column sums wrong: {out}'
    )

    # --- AGREEMENT with torch.autograd for a broadcast multiply ---
    # y = (a * b).sum(); a has shape (1,4), b has shape (3,4) — a is broadcast.
    a = t.randn(1, 4, generator=t.Generator().manual_seed(7), requires_grad=True)
    b = t.randn(3, 4, generator=t.Generator().manual_seed(8), requires_grad=True)
    y = (a * b).sum()
    y.backward()
    # our manual: grad_out = ones_like(a*b) = (3,4); dL/da = grad_out * b
    grad_out = t.ones(3, 4)
    raw_grad_a = grad_out * b.detach()  # shape (3,4) — wrong shape for `a`
    our_grad_a = unbroadcast(raw_grad_a, a.detach())
    assert our_grad_a.shape == a.shape, f'shape post-unbroadcast: {our_grad_a.shape}'
    assert t.allclose(our_grad_a, a.grad, atol=1e-5), (
        f'unbroadcast disagrees with autograd: ours={our_grad_a}, ref={a.grad}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def unbroadcast(grad: Tensor, original: Tensor) -> Tensor:
    # Step 1: peel leading axes broadcasting added.
    # If grad has more dims than original, the EXTRA ones must be on
    # the left (broadcasting always prepends 1s), so sum dim=0 repeatedly.
    while grad.ndim > original.ndim:
        grad = grad.sum(dim=0)
    # Step 2: collapse size-1 axes that were expanded.
    # For each axis where original has size 1 but grad doesn't, sum it
    # out with keepdim=True so we keep the size-1 axis instead of dropping it.
    for i, size in enumerate(original.shape):
        if size == 1 and grad.shape[i] != 1:
            grad = grad.sum(dim=i, keepdim=True)
    return grad
```

**Step 1 before step 2 — the order matters.** After step 1, `grad.ndim == original.ndim`, so axes line up positionally for step 2. If you tried to do the size-1 collapse first you'd be indexing into mismatched dims.

**Why `keepdim=True` in step 2.** Without it, `grad.sum(dim=i)` DROPS that axis, leaving `grad.ndim < original.ndim`. Then the result wouldn't match `original.shape` (which still has a size-1 axis at position `i`). With `keepdim=True`, the axis stays as size 1 — exactly matching.

**Where this lives in the codebase.** Every binary back fn that supports broadcasting wraps its result: `return unbroadcast(grad_out * y, x)` for `multiply_back0`, etc. The wrapper is the only way the autograd layer survives broadcasting — otherwise a broadcasted add of a `(1,4)` bias to a `(B,4)` batch would try to store a `(B,4)` grad on a `(1,4)` parameter and crash.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()